In [0]:

import unicodedata
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, trim, udf,
    datediff, to_date, udf, DataFrame
)
from pyspark.sql.types import IntegerType, DoubleType, StringType, DateType
from pyspark.sql.utils import AnalysisException

# Dicionário de meses
meses = {
    "janeiro": "01", "fevereiro": "02", "março": "03", "abril": "04",
    "maio": "05", "junho": "06", "julho": "07", "agosto": "08",
    "setembro": "09", "outubro": "10", "novembro": "11", "dezembro": "12"
}

# Dicionário de estados
estados_validos = {
    "acre": "Acre", "ac": "Acre",
    "alagoas": "Alagoas", "al": "Alagoas",
    "amapa": "Amapá", "ap": "Amapá",
    "amazonas": "Amazonas", "am": "Amazonas",
    "bahia": "Bahia", "ba": "Bahia",
    "ceara": "Ceará", "ce": "Ceará",
    "distrito federal": "Distrito Federal", "df": "Distrito Federal",
    "espirito santo": "Espírito Santo", "es": "Espírito Santo",
    "goias": "Goiás", "go": "Goiás",
    "maranhao": "Maranhão", "ma": "Maranhão",
    "mato grosso": "Mato Grosso", "mt": "Mato Grosso",
    "mato grosso do sul": "Mato Grosso do Sul", "ms": "Mato Grosso do Sul",
    "minas gerais": "Minas Gerais", "mg": "Minas Gerais",
    "para": "Pará", "pa": "Pará",
    "paraiba": "Paraíba", "pb": "Paraíba",
    "parana": "Paraná", "pr": "Paraná",
    "pernambuco": "Pernambuco", "pe": "Pernambuco",
    "piaui": "Piauí", "pi": "Piauí",
    "rio de janeiro": "Rio de Janeiro", "rj": "Rio de Janeiro",
    "rio grande do norte": "Rio Grande do Norte", "rn": "Rio Grande do Norte",
    "rio grande do sul": "Rio Grande do Sul", "rs": "Rio Grande do Sul",
    "rondonia": "Rondônia", "ro": "Rondônia",
    "roraima": "Roraima", "rr": "Roraima",
    "santa catarina": "Santa Catarina", "sc": "Santa Catarina",
    "sao paulo": "São Paulo", "sp": "São Paulo",
    "sergipe": "Sergipe", "se": "Sergipe",
    "tocantins": "Tocantins", "to": "Tocantins"
}

def converter_data(data_str):
    try:
        partes = data_str.split(",")[1].strip().split(" de ")
        dia = partes[0].zfill(2)
        mes = meses[partes[1].lower()]
        ano = partes[2]
        return f"{ano}-{mes}-{dia}"
    except:
        return None
    
converter_data_udf = udf(converter_data, StringType()) # Registro da função como UDF

def normalizar_estado(valor):
        if not valor:
            return "Estado inexistente"
        valor = valor.strip().lower()
        valor = unicodedata.normalize('NFKD', valor).encode('ASCII', 'ignore').decode('utf-8') # Remove acentos
        return estados_validos.get(valor, "Estado inexistente")

normalizar_estado_udf = udf(normalizar_estado, StringType()) # Registro da função como UDF

def transform_to_silver(df_bronze: DataFrame) -> DataFrame:
    
    # Remove espaços em colunas do tipo string
    df_limpo = df_bronze.select([
    trim(col(c)).alias(c) if t == "string" else col(c)
    for c, t in df_bronze.dtypes
    ])

    # Conversão de tipos e renomeações de colunas conforme o modelo
    df_corrigido = df_limpo \
        .withColumn("id_envio", col("id_envio").cast(IntegerType())) \
        .withColumn("corredor_de_armazenagem", col("corredor_de_armazenagem").cast(StringType())) \
        .withColumn("metodo_de_envio", col("metodo_de_envio").cast(StringType())) \
        .withColumn("ligacoes_do_cliente", col("ligações_do_cliente").cast(IntegerType())) \
        .withColumn("avaliacao_do_cliente", col("avaliação_do_cliente").cast(IntegerType())) \
        .withColumn("preco", col("preço").cast(DoubleType())) \
        .withColumn("qtd_itens", col("qtd_itens").cast(IntegerType())) \
        .withColumn("importancia", col("importancia").cast(StringType())) \
        .withColumn("genero", col("genero").cast(StringType())) \
        .withColumn("desconto", col("desconto").cast(DoubleType())) \
        .withColumn("peso_em_gramas", col("peso_g").cast(DoubleType())) \
        .withColumn("chegou_no_tempo", col("Chegou_no_tempo").cast(IntegerType())) \
        .withColumn("destino", normalizar_estado_udf(col("destino")).cast(StringType())) \
        .withColumn("data_do_envio", converter_data_udf(col("DataEnvio")).cast(DateType())) \
        .withColumn("data_da_entrega", converter_data_udf(col("DataEntrega")).cast(DateType())) \
        .withColumn("avaliacao_da_entrega", col("avaliacaoEntrega").cast(IntegerType()))

    # Cálculo de tempo de entrega em dias úteis
    df_corrigido = df_corrigido.withColumn(
        "tempo_entrega_dias",
        datediff(to_date(col("data_da_entrega")), to_date(col("data_do_envio")))
    )

    # Preenchimento de valores nulos com padrões
    df_tratado = df_corrigido.fillna({
        "importancia": "desconhecida",
        "genero": "Não informado",
        "destino": "desconhecido",
    })

    # Conversão de binário para string
    df_tratado = df_tratado.withColumn(
        "chegou_no_tempo",
        when(col("Chegou_no_tempo") == 1, "Sim").otherwise("Não")
    )

    # Padronização dos valores
    df_tratado = df_tratado.withColumn(
        "genero",
        when(col("genero").rlike("(?i)^m(asc)?"), "Masculino")
        .when(col("genero").rlike("(?i)^f(em)?"), "Feminino")
        .otherwise("Não informado")
    )

    # Padronização dos valores
    df_tratado = df_tratado.withColumn(
        "importancia",
        when(col("importancia").rlike("(?i)^low"), "Baixa")
        .when(col("importancia").rlike("(?i)^medium"), "Média")
        .when(col("importancia").rlike("(?i)^high"), "Alta")
        .otherwise("Desconhecida")
    )

    # Remove duplicidades com base no ID de envio (chave primária)
    df_silver = df_tratado.dropDuplicates(["id_envio"])

    df_silver = df_silver \
        .withColumn("camada", lit("silver")) \
        .withColumn("dt_processamento", current_timestamp().cast(DateType())) # Adiciona coluna com timestamp de processamento para rastreabilidade

    # Seleciona e organiza as colunas finais para gravação
    df_silver = df_silver.select("id_envio", "corredor_de_armazenagem","metodo_de_envio","ligacoes_do_cliente",
        "avaliacao_do_cliente","preco","qtd_itens","importancia","genero","desconto","peso_em_gramas",
        "chegou_no_tempo","destino","data_do_envio","data_da_entrega","avaliacao_da_entrega","tempo_entrega_dias",
        "camada","dt_processamento")

    return df_silver


In [0]:
def validar_silver(df_silver, contagem_esperada=None):
    print("Iniciando suíte de validação da camada Silver...")
    print("=" * 60)

    try:
        # Validação do schema: compara o schema atual do DataFrame com o esperado
        print("✅ Validação de Schema...")
        schema_esperado = {
            'id_envio': 'int', 'corredor_de_armazenagem': 'string', 'metodo_de_envio': 'string',
            'ligacoes_do_cliente': 'int', 'avaliacao_do_cliente': 'int', 'preco': 'double',
            'qtd_itens': 'int', 'importancia': 'string', 'genero': 'string', 'desconto': 'double',
            'peso_em_gramas': 'double', 'chegou_no_tempo': 'string', 'destino': 'string',
            'data_do_envio': 'date', 'data_da_entrega': 'date', 'avaliacao_da_entrega': 'int',
            'tempo_entrega_dias': 'int', 'camada': 'string', 'dt_processamento': 'date'
        }
        schema_atual = {field.name: field.dataType.simpleString() for field in df_silver.schema.fields}
        assert schema_atual == schema_esperado
        print("Schema está correto.")
    except AssertionError:
        print("Schema incorreto.")
        print("Esperado:", schema_esperado) # Exibe informações detalhadas caso o schema esteja incorreto
        print("Recebido:", schema_atual)
        return False
        raise

        # Verificação de valores nulos nos campos obrigatórios
    campos_obrigatorios = ["id_envio", "corredor_de_armazenagem","metodo_de_envio","ligacoes_do_cliente",
        "avaliacao_do_cliente","preco","qtd_itens","importancia","genero","desconto","peso_em_gramas",
        "chegou_no_tempo","destino","data_do_envio","data_da_entrega","avaliacao_da_entrega","tempo_entrega_dias",
        "camada","dt_processamento"]
    for campo in campos_obrigatorios:
        nulls = df_silver.filter(col(campo).isNull()).count()
        if nulls > 0:
            print(f"Campo obrigatório '{campo}' possui {nulls} valores nulos.")
            return False

    print(f"Campos obrigatórios não possuem valores nulos.")

    try:
        # Validações básicas de regras de negócio
        print("Validação de regras de negócio")
        df_silver.select("importancia").distinct().show()
        df_silver.select("genero").distinct().show()
        df_silver.select("chegou_no_tempo").distinct().show()
        print("Regras básicas aplicadas com sucesso.")
    except Exception as e:
        print("Falha ao validar regras de negócio:", str(e))
        return False
        raise

    print("=" * 60)
    print("Validações concluídas.")
    return True


In [0]:
from pyspark.sql import SparkSession   

spark = SparkSession.builder.getOrCreate()

# Leitura da tabela Bronze
df_bronze = spark.table("grao_direto.default.tabela_bronze_grain_logistic")

# Aplica transformações e limpezas 
silver = transform_to_silver(df_bronze)   

# Executa validações estruturais e de regras de negócio
deu_certo = validar_silver(silver)

# Caso todas as validações sejam aprovadas, persiste os dados em formato Delta na camada Silver
try:
    if deu_certo:
        silver.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable("grao_direto.default.tabela_silver_grain_logistic")
        print("Camada Silver finalizada com sucesso!")
    else:
        print("Falha na validação Silver. Verifique os erros acima para mais detalhes.")
except Exception as e:
    print(f"Erro ao salvar a tabela Silver: {str(e)}")